# Copper V24 — SA-LSTM Missing Holdout Only (Kaggle One-Shot)

**Run All:** 12 fitted jobs = 2 models × 6 missing (feature, split, strategy) combinations. **No SA search, no rerun of existing completed configurations.**

| Feature set | Train ratio | Missing strategies | Models |
|---|---|---|---|
| FS06_WTI_CU (`Copper, Aluminium, Tin, WTI`) | 80% | Rolling 756, Recency Weighting | Paper SA-LSTM, Reoptimized SA-LSTM |
| FS04_METALS_CU (`Copper, Aluminium, Tin, Silver, Gold`) | 92.5%, 95% | Rolling 756, Recency Weighting | Paper SA-LSTM, Reoptimized SA-LSTM |

**Fixed previously selected HP:** paper: `LSTM(39,111), lr=5e-5`; legacy optimized cache V21: `LSTM(22,83), lr=0.000655`; WTI fresh SA V22: `LSTM(47,127), lr=5.0478806754686805e-05`.

Output: `copper_v24_sa_missing_results.zip` containing metrics, predictions, HP provenance, manifest, and plots. Same method as SA-series V21/V22 (input 22 sessions, horizon one, same scaler/splitting scheme). Results are *new experimental values* only after Kaggle executes this notebook; no values are prefilled.

In [ ]:
from pathlib import Path
import os, gc, json, time, math, hashlib, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf

LOOKBACK=22; HORIZON=1; SEED=42; BATCH_SIZE=64
CFG={'epochs':100,'earlystop':True}
TARGET='Copper'
EXPECTED_SHA='9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600'
PAPER_SA_LSTM_HP={'units_1':39,'units_2':111,'learning_rate':5e-5}
REOPT_HP={
    'legacy14_with_copper':{'units_1':22,'units_2':83,'learning_rate':0.000655},
    'wti_replacement_with_copper':{'units_1':47,'units_2':127,'learning_rate':5.0478806754686805e-05},
}
FEATURE_SETS={
    'legacy14_with_copper':['Copper','Aluminium','Tin','Silver','Gold'],
    'wti_replacement_with_copper':['Copper','Aluminium','Tin','WTI'],
}
FS_CODES={'legacy14_with_copper':'FS04_METALS_CU','wti_replacement_with_copper':'FS06_WTI_CU'}
JOBS=[
    ('wti_replacement_with_copper',0.8,'rolling_756'),
    ('wti_replacement_with_copper',0.8,'recency_weighted'),
    ('legacy14_with_copper',0.925,'rolling_756'),
    ('legacy14_with_copper',0.925,'recency_weighted'),
    ('legacy14_with_copper',0.95,'rolling_756'),
    ('legacy14_with_copper',0.95,'recency_weighted'),
]
METHODS=('paper_sa_lstm','reopt_sa_lstm')
RUN_STARTED=time.time()
OUT=Path('/kaggle/working/copper_v24_sa_missing') if Path('/kaggle/working').exists() else Path('/mnt/data/copper_v24_sa_missing')
OUT.mkdir(parents=True,exist_ok=True)
CANDIDATES=[
    Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('dataset_9_commodities_daily.csv'),
]
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    try:tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError:pass
print('GPU:',tf.config.list_physical_devices('GPU'),'Jobs:',len(JOBS)*len(METHODS),'Output:',OUT)

In [ ]:
DATA_PATH=next((p for p in CANDIDATES if p.is_file()),None)
if DATA_PATH is None and Path('/kaggle/input').exists():
    matches=list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(matches)==1:DATA_PATH=matches[0]
    elif len(matches)>1:raise RuntimeError('Multiple dataset files found; select the same source used in V20–V23 explicitly.')
if DATA_PATH is None:raise FileNotFoundError('Attach Kaggle dataset containing dataset_9_commodities_daily.csv')
dataset_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert dataset_sha256==EXPECTED_SHA, ('Dataset SHA differs from V20–V23. Stop to avoid invalid comparisons:',dataset_sha256)
raw=pd.read_csv(DATA_PATH)
required=['Date','Copper','Aluminium','Tin','Silver','Gold','WTI']
missing=[c for c in required if c not in raw.columns]
if missing:raise KeyError(f'Missing columns: {missing}')
raw['Date']=pd.to_datetime(raw['Date'],errors='raise')
assert not raw['Date'].duplicated().any(), 'Duplicate Date'
for c in required[1:]:raw[c]=pd.to_numeric(raw[c],errors='raise')
data=raw.sort_values('Date').reset_index(drop=True)
assert data[required].notna().all().all(),'Unexpected missing values: would change test alignment'
assert len(data)==2710 and str(data.Date.iloc[0].date())=='2016-01-13' and str(data.Date.iloc[-1].date())=='2026-11-13'
assert (data[required[1:]]>0).all().all()
print(f'VERIFIED {len(data)} sessions: {data.Date.iloc[0].date()}–{data.Date.iloc[-1].date()}, SHA256={dataset_sha256}')

In [ ]:
def make_context(frame, ratio, scheme='expanding', feature_set='wti_replacement_with_copper', val_fraction=0.10):
    n = len(frame)
    split = int(n * ratio)
    val_start = int(split * (1.0 - val_fraction))
    fit_start = 0 if scheme != 'rolling_756' else max(0, val_start - 756)
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 25 < val_start < split < n
    assert val_start - fit_start > LOOKBACK + 10
    # Train-only scaling: no future statistics leak into scaler.
    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])
    x = xsc.transform(frame[cols]).astype('float32')
    y = ysc.transform(frame[[TARGET]]).ravel().astype('float32')
    prev = np.r_[np.nan, y[:-1]]
    def block(start, end, restrict_train=False):
        indices = np.arange(max(start, LOOKBACK), end)
        if restrict_train:
            indices = indices[indices-LOOKBACK >= fit_start]
        X = np.stack([x[t-LOOKBACK:t] for t in indices]).astype('float32')
        return {'idx': indices, 'X': X, 'y': y[indices],
                'prev': prev[indices].astype('float32'),
                'date': frame.Date.iloc[indices].to_numpy()}
    tr = block(fit_start, val_start, True)
    va = block(val_start, split)
    te = block(split, n)
    assert tr['idx'][-1] < va['idx'][0] < te['idx'][0]
    assert xsc.n_samples_seen_ == val_start - fit_start
    assert len(tr['X']) > 0 and tr['X'].shape[1:] == (LOOKBACK, len(cols))
    # Last X timestamp is always t-1. Verify Copper only if included.
    if 'Copper' in cols:
        copper_col = cols.index('Copper')
        copper_raw_in_x = xsc.inverse_transform(x[tr['idx']-1])[:, copper_col]
        copper_true = frame.iloc[tr['idx']-1]['Copper'].to_numpy()
        assert np.allclose(copper_raw_in_x, copper_true, atol=1e-4)
    return {'train': tr, 'val': va, 'test': te, 'xsc': xsc, 'ysc': ysc,
            'features': cols, 'feature_set': feature_set, 'horizon': 1,
            'ratio': float(ratio), 'scheme': scheme,
            'fit_start': fit_start, 'val_start': val_start, 'test_start': split}

In [ ]:
PAPER_SA_LSTM_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}

def build_baseline_lstm(input_shape, hp=None):
    """Mô hình LSTM chuẩn không tối ưu SA (Protocol V20 baseline)."""
    model = tf.keras.Sequential(name='Baseline_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

def build_paper_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM với siêu tham số công bố trong bài báo Chen et al. (2023)."""
    model = tf.keras.Sequential(name='Paper_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='mse')
    return model

def build_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM nhận siêu tham số linh hoạt do SA tìm kiếm (chỉ H=1, HP tìm mới riêng cho từng feature set)."""
    h = hp or PAPER_SA_LSTM_HP
    u1 = int(h.get('units_1', 39))
    u2 = int(h.get('units_2', 111))
    lr = float(h.get('learning_rate', 5e-5))

    model = tf.keras.Sequential(name='Reoptimized_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(u1, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(u2, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse')
    return model

# Đặt alias tương thích ngược nếu có đoạn gọi tên riêng
build_sa_lstm_paper3 = build_sa_lstm
build_sa_lstm_legacy14 = build_sa_lstm

print('Đã khởi tạo xong các hàm dựng mô hình LSTM (bao gồm build_sa_lstm).')

def build_baseline_rnn(input_shape, hp=None):
    """SimpleRNN baseline with two recurrent layers and comparable head."""
    model = tf.keras.Sequential(name='Baseline_RNN')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.SimpleRNN(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.SimpleRNN(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
def reset_keras(seed=SEED):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(int(seed))

def sample_weights(length, scheme, half_life=252):
    if scheme != 'recency_weighted': return None
    ages = np.arange(length - 1, -1, -1, dtype='float64')
    w = np.exp(-np.log(2) * ages / half_life)
    return (w / w.mean()).astype('float32')

def evaluate_predictions(ctx, model_name, p_val, p_test, seed=SEED, extras=None):
    ysc = ctx['ysc']; va = ctx['val']; te = ctx['test']
    def original(z): return ysc.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel()

    # 1. Dịch ngược về giá gốc (USD)
    y_true = original(te['y'])
    pred = original(p_test)
    base = original(te['prev'])

    delta_pred = pred - base
    delta_true = y_true - base

    # 2. Tính các độ đo trên giá gốc (RAW - USD) khớp 100% với V20
    mae = float(mean_absolute_error(y_true, pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
    mse = float(mean_squared_error(y_true, pred))  # DỊCH NGƯỢC LẠI GIÁ GỐC (USD^2) - KHỚP V20
    base_mae = float(mean_absolute_error(y_true, base))
    improvement = float(100.0 * (base_mae - mae) / base_mae) if base_mae > 0 else 0.0

    # 3. Tính các độ đo trên thang đo chuẩn hóa (SCALED [0, 1])
    val_mse_scaled = float(mean_squared_error(va['y'], p_val))
    mse_scaled = float(mean_squared_error(te['y'], p_test)) if p_test is not None else np.nan

    # 4. Tính Directional Accuracy và các thống kê kiểm toán phần dư
    dir_acc = float(100.0 * np.mean(np.sign(delta_pred) == np.sign(delta_true))) if model_name != 'naive' else np.nan
    delta_pred_std = float(np.std(delta_pred))
    delta_true_std = float(np.std(delta_true))
    delta_pred_mean_abs = float(np.mean(np.abs(delta_pred)))
    pred_delta_corr = float(np.corrcoef(delta_pred, delta_true)[0, 1]) if delta_pred_std > 1e-12 else np.nan

    row = {
        'model': model_name,
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'scheme': ctx['scheme'],
        'ratio': ctx['ratio'],
        'seed': int(seed),
        'n_test': len(y_true),
        'mae': mae,
        'rmse': rmse,
        'mse': mse,                        # GIÁ GỐC DỊCH NGƯỢC (USD^2) - KHỚP V20
        'mse_scaled': mse_scaled,          # SCALED TRÊN TẬP TEST [0, 1]
        'mae_naive': base_mae,
        'mae_improvement_pct': improvement,
        'val_mse_scaled': val_mse_scaled,  # SCALED TRÊN TẬP VALIDATION [0, 1] - KHỚP V20
        'directional_accuracy_pct': dir_acc,
        'delta_pred_std_raw': delta_pred_std,
        'delta_true_std_raw': delta_true_std,
        'delta_pred_mean_abs_raw': delta_pred_mean_abs,
        'pred_delta_corr': pred_delta_corr,
        'test_start': str(pd.Timestamp(te['date'][0]).date()),
        'fit_start': str(data.Date.iloc[ctx['fit_start']].date())
    }
    if extras: row.update(extras)

    predictions = pd.DataFrame({
        'Date': te['date'],
        'ratio': ctx['ratio'],
        'scheme': ctx['scheme'],
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'model': model_name,
        'seed': int(seed),
        'actual': y_true,
        'pred': pred,
        'naive': base,
        'pred_delta': delta_pred,
        'actual_delta': delta_true
    })
    return row, predictions

def fit_predict_model(builder, ctx, hp=None, seed=SEED, epochs=None, patience=None, is_search=False, skip_test=False):
    reset_keras(seed)
    tr, va, te = ctx['train'], ctx['val'], ctx['test']
    model = builder(tr['X'].shape[1:], hp)

    cb = []
    if CFG['earlystop']:
        pat = patience if patience is not None else (4 if is_search else 9)
        cb.append(tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat, restore_best_weights=True))

    ep = epochs if epochs is not None else (CFG['search_epochs'] if is_search else CFG['epochs'])
    weights = sample_weights(len(tr['idx']), ctx['scheme'])

    start_t = time.perf_counter()
    hist = model.fit(
        tr['X'], tr['y'],
        validation_data=(va['X'], va['y']),
        epochs=ep,
        batch_size=BATCH_SIZE,
        sample_weight=weights,
        callbacks=cb,
        shuffle=False,
        verbose=0
    )
    fit_sec = time.perf_counter() - start_t

    pv = model.predict(va['X'], verbose=0).ravel()
    pt = model.predict(te['X'], verbose=0).ravel() if not skip_test else None
    val_mse = float(mean_squared_error(va['y'], pv))
    epochs_run = len(hist.history['loss'])

    del model
    reset_keras(seed)
    return {'val_pred': pv, 'test_pred': pt, 'val_mse': val_mse, 'seconds': fit_sec, 'epochs_run': epochs_run}

## Run the 12 missing SA-LSTM training jobs

Each run is fitted with original fixed HP of its feature group; training uses chronological train/validation split and test is used only for reporting. **Recency samples are weighted with half-life 252 sessions**, while Rolling retains 756 recent sessions. The model is refitted once per holdout combination, not retrained per test day.

In [ ]:
FINGERPRINT=hashlib.sha256(json.dumps({'sha':dataset_sha256,'jobs':JOBS,'methods':METHODS,'hp':REOPT_HP,'seed':SEED,'cfg':CFG,'lookback':LOOKBACK},sort_keys=True).encode()).hexdigest()[:16]
RESULT_PATH=OUT/'holdout_missing_results.csv'
PRED_PATH=OUT/'holdout_missing_predictions.csv'
# Run All means a clean 12-job experiment, never mix stale rows or results from a partial run.
for path in (RESULT_PATH,PRED_PATH):
    if path.exists():path.unlink()
rows=[]
for j,(feature_set,ratio,scheme) in enumerate(JOBS,1):
    ctx=make_context(data,ratio,scheme,feature_set)
    for model_name in METHODS:
        hp=PAPER_SA_LSTM_HP if model_name=='paper_sa_lstm' else REOPT_HP[feature_set]
        builder=build_paper_sa_lstm if model_name=='paper_sa_lstm' else build_sa_lstm
        result=fit_predict_model(builder,ctx,hp=hp,seed=SEED)
        extras={'epochs_run':result['epochs_run'],'fit_seconds':result['seconds'],
                'hp_config':json.dumps(hp,sort_keys=True),'hp_origin':('paper_fixed' if model_name=='paper_sa_lstm' else ('V21_legacy_cached' if feature_set=='legacy14_with_copper' else 'V22_WTI_SA_16_trials')),
                'source':'V24','fs_code':FS_CODES[feature_set]}
        row,pred=evaluate_predictions(ctx,model_name,result['val_pred'],result['test_pred'],seed=SEED,extras=extras)
        y=pred['actual'].to_numpy(float);yh=pred['pred'].to_numpy(float)
        row['mape_pct']=float(np.mean(np.abs((y-yh)/y))*100)
        row['r2']=float(r2_score(y,yh))
        row['fingerprint']=FINGERPRINT
        pred['source']='V24';pred['fs_code']=FS_CODES[feature_set];pred['fingerprint']=FINGERPRINT
        rows.append(row)
        pd.DataFrame(rows).to_csv(RESULT_PATH,index=False)
        pred.to_csv(PRED_PATH,mode='a',header=not PRED_PATH.exists(),index=False)
        print(f'[{len(rows):02}/12] {FS_CODES[feature_set]} ratio={ratio:g} {scheme} {model_name} '
              f'MAE={row["mae"]:.6f} RMSE={row["rmse"]:.6f} epochs={row["epochs_run"]}',flush=True)
assert len(rows)==12, f'Expected 12 results, got {len(rows)}'

## Audit and final export

Raw prediction rows must equal `2 × (2×542 + 4×204 + 4×136)`? No: each of the six feature/split/strategy settings is fitted twice, giving `4×542 + 4×204 + 4×136 = 3,528` prediction rows. Test dates are checked against each ratio. No results for older configurations are manufactured.

In [ ]:
res=pd.read_csv(RESULT_PATH)
pred=pd.read_csv(PRED_PATH)
assert len(res)==12 and len(pred)==3528,(len(res),len(pred))
assert res['fingerprint'].eq(FINGERPRINT).all() and pred['fingerprint'].eq(FINGERPRINT).all()
assert not pred.duplicated(['ratio','scheme','feature_set','model','Date','seed']).any()
for key,g in pred.groupby(['ratio','scheme','feature_set','model']):
    rr=res[(res['ratio']==key[0])&(res['scheme']==key[1])&(res['feature_set']==key[2])&(res['model']==key[3])]
    assert len(rr)==1
    ma=float(np.mean(np.abs(g.actual-g.pred)))
    rm=float(np.sqrt(np.mean((g.actual-g.pred)**2)))
    assert np.isclose(ma,rr.iloc[0].mae,rtol=1e-6,atol=1e-7)
    assert np.isclose(rm,rr.iloc[0].rmse,rtol=1e-6,atol=1e-7)
    assert len(g)==len(data)-int(len(data)*key[0])
res['fs_code']=res.feature_set.map(FS_CODES)
view=res[['fs_code','ratio','scheme','model','mae','rmse','mape_pct','r2','n_test']].sort_values(['ratio','fs_code','scheme','mae'])
print(view.to_string(index=False))
fig,axs=plt.subplots(1,2,figsize=(14,5))
for ax,(fs,sub) in zip(axs,res.groupby('feature_set',sort=False)):
    labs=[f'{r.ratio:g}/{r.scheme.replace("_weighted","").replace("rolling_756","roll")} / {r.model.replace("_sa_lstm", "")}' for r in sub.itertuples()]
    ax.bar(range(len(sub)),sub.mae)
    ax.set_xticks(range(len(sub)),labs,rotation=65,ha='right',fontsize=8)
    ax.set_title(FS_CODES[fs]);ax.set_ylabel('MAE (original Copper price units)')
fig.tight_layout();fig.savefig(OUT/'holdout_missing_mae.png',dpi=160,bbox_inches='tight');plt.close(fig)
manifest={
 'version':'V24_SA_gapfill','source_dataset':str(DATA_PATH),'sha256':dataset_sha256,
 'date_range':[str(data.Date.iloc[0].date()),str(data.Date.iloc[-1].date())],
 'rows':len(data),'features':FEATURE_SETS,'fs_codes':FS_CODES,'jobs':JOBS,
 'models':list(METHODS),'paper_hp':PAPER_SA_LSTM_HP,'reoptimized_hp':REOPT_HP,
 'hyperparameter_search_rerun':False,'hp_sources':{'FS04_METALS_CU':'V21 cached','FS06_WTI_CU':'V22 fresh SA 16 trials'},
 'epochs_max':CFG['epochs'],'early_stopping_patience':9,'batch_size':BATCH_SIZE,
 'seed':SEED,'lookback':LOOKBACK,'horizon':HORIZON,'train_validation_fraction':0.1,
 'rolling_length_sessions':756,'recency_half_life_sessions':252,
 'n_holdout_jobs':len(res),'n_prediction_rows':len(pred),'run_fingerprint':FINGERPRINT,
 'no_test_tuning':True,'validation_scaling_only':True,'runtime_seconds':time.time()-RUN_STARTED
}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(OUT/'README_RESULTS.md').write_text('# V24 SA gap fill\n\n12 fitted runs; no SA search; HP from V21/V22; models use fixed Paper or reoptimized SA configurations.\n\nFiles: holdout_missing_results.csv, holdout_missing_predictions.csv, holdout_missing_mae.png, experiment_manifest.json.\n')
ZIP=OUT.parent/'copper_v24_sa_missing_results.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():z.write(p,arcname=f'{OUT.name}/{p.name}')
print('SUCCESS:',ZIP,'bytes:',ZIP.stat().st_size)